# SD-MAC — API Demo (in-process, no live server)

**Component:** SD-MAC (Sector-Wide Deployable Modular Analytics Commons)

This notebook exercises **all seven** record endpoints of the SD-MAC FastAPI application plus the operational `/healthz` probe, using `fastapi.testclient.TestClient` — so no live server is required. The app serves entirely from bundled synthetic fixtures and schema-registry manifests.

> **Synthetic data only.** Every dataset used in this notebook is *synthetic illustrative data generated for demonstration*. It is **NOT** real agency data and is **NOT** derived from any proprietary or employer source. The notebook performs no network access and uses only public-style, open-source components.

In [1]:
import sys
from pathlib import Path

# Put the oefaf-platform repository root on sys.path so the platform
# component packages (gea, cricat, sdmac, shared) import cleanly no matter
# what working directory this notebook is executed from.
_REPO_ROOT = Path.cwd()
while not (_REPO_ROOT / 'sdmac' / 'schema_registry').is_dir():
    if _REPO_ROOT == _REPO_ROOT.parent:
        raise RuntimeError('Could not locate the oefaf-platform repo root.')
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
print('repo root located:', _REPO_ROOT.name)

repo root located: oefaf-platform


## 0. Boot the app in-process

`TestClient(app)` wraps the ASGI app and dispatches requests in-process.

In [2]:
import json

from fastapi.testclient import TestClient
from sdmac.api.main import app

client = TestClient(app)

def show(label, resp):
    body = resp.json()
    n = len(body) if isinstance(body, list) else 1
    print(f'{label}: HTTP {resp.status_code}  ({n} record(s))')
    return body

health = show('GET /healthz', client.get('/healthz'))
print(json.dumps(health, indent=2))

GET /healthz: HTTP 200  (1 record(s))
{
  "status": "ok",
  "events_loaded": 12,
  "forecasts_loaded": 4,
  "scenarios_loaded": 3,
  "modules_loaded": 5
}


## 1. GEA — `GET /v1/events` (list, with filters)

Returns the compact event projection. We also demonstrate the `min_severity` filter.

In [3]:
events = show('GET /v1/events', client.get('/v1/events'))
print(json.dumps(events[:2], indent=2))

filtered = show(
    'GET /v1/events?min_severity=0.3',
    client.get('/v1/events', params={'min_severity': 0.3}),
)
first_event_id = events[0]['event_id']
print('first event_id:', first_event_id)

GET /v1/events: HTTP 200  (12 record(s))
[
  {
    "event_id": "GEA-EVT-0000",
    "detected_at_utc": "2026-01-05T23:00:00Z",
    "severity_score": 0.828,
    "public_evidence_refs": [
      "https://example.org/public-evidence/crude_oil/0000/0",
      "https://example.org/public-evidence/crude_oil/0000/1",
      "https://example.org/public-evidence/crude_oil/0000/2"
    ]
  },
  {
    "event_id": "GEA-EVT-0001",
    "detected_at_utc": "2026-01-16T02:00:00Z",
    "severity_score": 0.971,
    "public_evidence_refs": [
      "https://example.org/public-evidence/natural_gas/0001/0",
      "https://example.org/public-evidence/natural_gas/0001/1"
    ]
  }
]
GET /v1/events?min_severity=0.3: HTTP 200  (10 record(s))
first event_id: GEA-EVT-0000


## 2. GEA — `GET /v1/events/{event_id}` (full record)

In [4]:
full_event = show(
    f'GET /v1/events/{first_event_id}',
    client.get(f'/v1/events/{first_event_id}'),
)
print(json.dumps(full_event, indent=2))

GET /v1/events/GEA-EVT-0000: HTTP 200  (1 record(s))
{
  "event_id": "GEA-EVT-0000",
  "detected_at_utc": "2026-01-05T23:00:00Z",
  "commodity": "crude_oil",
  "region_iso": "SAU",
  "source_categories": [
    "weather"
  ],
  "severity_score": 0.828,
  "confidence": 0.984,
  "public_evidence_refs": [
    "https://example.org/public-evidence/crude_oil/0000/0",
    "https://example.org/public-evidence/crude_oil/0000/1",
    "https://example.org/public-evidence/crude_oil/0000/2"
  ]
}


## 3. CRICAT — `GET /v1/forecasts/{iso_region}`

In [5]:
forecasts = show(
    'GET /v1/forecasts/PJM',
    client.get('/v1/forecasts/PJM'),
)
print(json.dumps(forecasts[0], indent=2))

GET /v1/forecasts/PJM: HTTP 200  (2 record(s))
{
  "forecast_id": "CRICAT-FC-0000",
  "iso_region": "PJM",
  "forecast_issued_at_utc": "2026-03-12T00:00:00Z",
  "target_window_start_utc": "2026-03-13T00:00:00Z",
  "target_window_end_utc": "2026-03-14T00:00:00Z",
  "predicted_load_mw": 71290.1,
  "prediction_interval_low_mw": 68438.5,
  "prediction_interval_high_mw": 74141.7,
  "model_id": "cricat-gbr-dayahead-v0.1",
  "input_data_sources": [
    "synthetic_load_history",
    "synthetic_weather",
    "calendar_features"
  ]
}


## 4. CRICAT — `POST /v1/scenarios` then `GET /v1/scenarios/{id}`

Demonstrates that a POST-created scenario persists in the in-memory store and is retrievable by the GET endpoint (round-trip).

In [6]:
create_resp = client.post(
    '/v1/scenarios',
    json={
        'scenario_label': 'demo_winter_cold_snap_pjm',
        'stress_drivers': ['cold_snap', 'fuel_constraint'],
        'regions': ['PJM'],
        'time_horizon_hours': 48,
    },
)
created = show('POST /v1/scenarios', create_resp)
scenario_id = created['scenario_id']
print(json.dumps(created, indent=2))

fetched = show(
    f'GET /v1/scenarios/{scenario_id}',
    client.get(f'/v1/scenarios/{scenario_id}'),
)
assert fetched['scenario_id'] == scenario_id, 'round-trip id mismatch'
print('POST -> GET scenario round-trip OK:', scenario_id)

POST /v1/scenarios: HTTP 201  (1 record(s))
{
  "scenario_id": "CRICAT-SCN-0003",
  "scenario_label": "demo_winter_cold_snap_pjm",
  "stress_drivers": [
    "cold_snap",
    "fuel_constraint"
  ],
  "time_horizon_hours": 48,
  "regions": [
    "PJM"
  ],
  "assumed_demand_mw": 91720.7,
  "assumed_available_capacity_mw": 79317.7,
  "reserve_margin_pct": -13.52,
  "probability_of_stress": 0.884,
  "data_provenance": [
    "https://example.org/public-provenance/demo_winter_cold_snap_pjm/0",
    "https://example.org/public-provenance/demo_winter_cold_snap_pjm/1"
  ]
}
GET /v1/scenarios/CRICAT-SCN-0003: HTTP 200  (1 record(s))
POST -> GET scenario round-trip OK: CRICAT-SCN-0003


## 5. SD-MAC — `GET /v1/modules` then `GET /v1/modules/{id}`

In [7]:
modules = show('GET /v1/modules', client.get('/v1/modules'))
for m in modules:
    print(f"  {m['module_id']:<26} {m['platform_component']:<8} {m['validation_status']}")

first_module_id = modules[0]['module_id']
full_module = show(
    f'GET /v1/modules/{first_module_id}',
    client.get(f'/v1/modules/{first_module_id}'),
)
print(json.dumps(full_module, indent=2))

GET /v1/modules: HTTP 200  (5 record(s))
  cricat-grid-modeling       cricat   draft
  cricat-load-forecasting    cricat   draft
  gea-scoring                gea      draft
  sdmac-api                  sdmac    draft
  shared-utilities           shared   draft
GET /v1/modules/cricat-grid-modeling: HTTP 200  (1 record(s))
{
  "module_id": "cricat-grid-modeling",
  "module_name": "CRICAT Grid Modeling",
  "platform_component": "cricat",
  "license": "MIT",
  "maintainers": [
    "<JING_WEN_TO_FILL: maintainer handle or institutional affiliation>"
  ],
  "public_dependencies": [
    "numpy==1.26.4",
    "pandas==2.2.2"
  ],
  "data_inputs": [
    "synthetic_demand_assumptions",
    "synthetic_capacity_assumptions",
    "synthetic_stress_drivers"
  ],
  "outputs": [
    "capacity_allocation_scenario"
  ],
  "validation_status": "draft",
  "documentation_url": "<JING_WEN_TO_FILL: documentation URL>"
}


## 6. Summary — all seven record endpoints exercised

In [8]:
checks = [
    ('GET  /v1/events', client.get('/v1/events').status_code),
    (f'GET  /v1/events/{first_event_id}', client.get(f'/v1/events/{first_event_id}').status_code),
    ('GET  /v1/forecasts/PJM', client.get('/v1/forecasts/PJM').status_code),
    ('POST /v1/scenarios', create_resp.status_code),
    (f'GET  /v1/scenarios/{scenario_id}', client.get(f'/v1/scenarios/{scenario_id}').status_code),
    ('GET  /v1/modules', client.get('/v1/modules').status_code),
    (f'GET  /v1/modules/{first_module_id}', client.get(f'/v1/modules/{first_module_id}').status_code),
    ('GET  /healthz', client.get('/healthz').status_code),
]
for ep, code in checks:
    print(f'{code}  {ep}')
assert all(code in (200, 201) for _, code in checks), 'an endpoint did not return 2xx'
print('\nAll seven record endpoints (+ /healthz) returned 2xx.')

200  GET  /v1/events
200  GET  /v1/events/GEA-EVT-0000
200  GET  /v1/forecasts/PJM
201  POST /v1/scenarios
200  GET  /v1/scenarios/CRICAT-SCN-0003
200  GET  /v1/modules
200  GET  /v1/modules/cricat-grid-modeling
200  GET  /healthz

All seven record endpoints (+ /healthz) returned 2xx.


---

**Recap.** All seven record endpoints plus `/healthz` were exercised in-process via `TestClient`, including a POST→GET scenario round-trip. Every response is synthetic illustrative data served from bundled fixtures; no live server, network access, or proprietary content is involved.